# 02 – EDA & Business Metrics
Basic exploration of the cleaned dataset. Looking at rating patterns, review lengths, helpfulness trends, and time-based behavior.


In [ ]:
USE DATABASE AMAZON_REVIEWS_DB;
USE SCHEMA ANALYTICS;
USE WAREHOUSE COMPUTE_WH;


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

session = get_active_session()
session


In [ ]:
reviews_df = (
    session.table("ANALYTICS.REVIEWS_CLEAN")
           .limit(200_000)   
           .to_pandas()
)

product_metrics_df = session.table("ANALYTICS.PRODUCT_METRICS").to_pandas()
daily_ratings_df   = session.table("ANALYTICS.DAILY_RATINGS").to_pandas()

print("REVIEWS_CLEAN:", reviews_df.shape)
print("PRODUCT_METRICS:", product_metrics_df.shape)
print("DAILY_RATINGS:", daily_ratings_df.shape)

reviews_df.head()


### Rating distribution
Seeing how ratings are spread across the dataset.


In [ ]:
print("Rating distribution:")
print(reviews_df["RATING"].value_counts().sort_index())

print("\nHelpfulness ratio summary:")
print(reviews_df["HELPFUL_RATIO"].describe())

print("\nReview text length (characters):")
reviews_df["TEXT_LENGTH"] = reviews_df["REVIEW_TEXT"].str.len()
print(reviews_df["TEXT_LENGTH"].describe())


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x="RATING", data=reviews_df, order=sorted(reviews_df["RATING"].unique()))
plt.title("Distribution of Review Ratings")
plt.xlabel("Rating (1–5 stars)")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()


### Review length vs rating
Checking whether longer reviews are tied to higher or lower ratings.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x="RATING", y="TEXT_LENGTH", data=reviews_df)
plt.title("Review Text Length by Rating")
plt.xlabel("Rating (1–5 stars)")
plt.ylabel("Review Length (characters)")
plt.tight_layout()
plt.show()


### Helpfulness ratio across ratings
Looking at how helpful reviews are at different rating levels.


In [ ]:
helpfulness_by_rating = (
    reviews_df
    .groupby("RATING", as_index=False)
    .agg(AVG_HELPFUL_RATIO=("HELPFUL_RATIO", "mean"))
)

plt.figure(figsize=(10, 5))
sns.barplot(x="RATING", y="AVG_HELPFUL_RATIO", data=helpfulness_by_rating)
plt.title("Average Helpfulness Ratio by Rating")
plt.xlabel("Rating (1–5 stars)")
plt.ylabel("Average Helpfulness Ratio")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

helpfulness_by_rating


### Rating trend over time
Checking if ratings changed over the years.


In [ ]:
daily_ratings_df["REVIEW_DATE"] = pd.to_datetime(daily_ratings_df["REVIEW_DATE"])

plt.figure(figsize=(14, 6))
sns.lineplot(x="REVIEW_DATE", y="AVG_DAILY_RATING", data=daily_ratings_df)
plt.title("Average Rating Over Time")
plt.xlabel("Date")
plt.ylabel("Average Daily Rating")
plt.tight_layout()
plt.show()

daily_ratings_df.head()


### Top rated products
Showing the best-performing products with readable names.


In [ ]:
product_name_stats = (
    reviews_df
    .groupby("PRODUCT_ID")
    .agg(
        AVG_RATING=("RATING", "mean"),
        N_REVIEWS=("RATING", "size"),
        PRODUCT_NAME=("SUMMARY", lambda x: x.value_counts().idxmax())
    )
    .reset_index()
)

product_name_stats = product_name_stats[product_name_stats["N_REVIEWS"] >= 50]

top_products = (
    product_name_stats
    .sort_values(["AVG_RATING", "N_REVIEWS"], ascending=[False, False])
    .head(10)
)

top_products


In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(
    x="PRODUCT_NAME",
    y="AVG_RATING",
    data=top_products
)

plt.title("Top 10 Products by Average Rating (Products with ≥ 50 Reviews)")
plt.xlabel("Product Name")
plt.ylabel("Average Rating")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
